In [1]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

library(dplyr)
library(edgeR)
library(limma)
library(data.table)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: limma


Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




In [2]:
data_source <- "GTEx_cortex_TPM_All_598_outliers_removed_ComBat_SMGEBTCH_corrected"

expr <- fread(paste0("data/", data_source, ".csv"), data.table=FALSE)

colnames(expr)[1] <- "Gene"

In [24]:
set_source <- "Claude_marker_genes"

In [3]:
top_mods_df <- fread("data/enrichments/GTEx_cortex_TPM_All_598_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.26_Modules_Claude_marker_genes_enrichments.csv", data.table=FALSE)

#### Get modules to regress out

In [7]:
top_mods_df$Cell_type <- gsub("/", "_", top_mods_df$Cell_type, fixed=TRUE)
top_mods_df$Cell_type <- gsub(" ", "_", top_mods_df$Cell_type, fixed=TRUE)

In [8]:
top_mods_df %>%
    arrange(Qval) %>%
    select(Cell_type, Qval)

Cell_type,Qval
<chr>,<dbl>
Micro_PVM,1.289199e-19
Oligo,1.871966e-15
Astro,2.015052e-15
All_GABAergic,1.351891e-14
VLMC,1.016071e-09
MGE_Class,4.150476e-08
Vip,9.531653e-08
Sst,6.938045e-07
L2_3_IT,1.600087e-06


In [9]:
sets <- c("Micro_PVM", "Oligo", "Astro", "All_GABAergic", "VLMC")

top_mods_df_subset <- top_mods_df[top_mods_df$Cell_type %in% sets,]

In [10]:
ME_df <- fread(top_mods_df_subset$ME_path[1], data.table=FALSE)
samples <- ME_df[,1]

ME_list <- lapply(seq_along(1:nrow(top_mods_df_subset)), function(i) {
    mod <- top_mods_df_subset$Module[i] 
    ME_df <- fread(top_mods_df_subset$ME_path[i], data.table=FALSE)
    ME_df[,grep(paste0("^", mod, "$"), colnames(ME_df))] 
})

ME_df <- data.frame(Sample=samples, do.call(cbind, ME_list))
colnames(ME_df)[-1] <- top_mods_df_subset$Cell_type

In [11]:
ME_df <- ME_df[match(colnames(expr)[-1], ME_df$Sample),]

all.equal(ME_df$Sample, colnames(expr)[-1])

[1] TRUE

In [13]:
# (optional) scale continuous covariates so coefficients behave nicely
scale_if_num <- function(x) if(is.numeric(x)) as.numeric(scale(x)) else x
ME_matrix <- apply(ME_df[,-1], 2, scale_if_num)

In [14]:
head(ME_matrix)

Micro_PVM,Oligo,Astro,All_GABAergic,VLMC
-0.02595624,-0.5909053,-1.0031370,1.86697930,-0.41521542
-0.60029400,-0.5656572,-1.3864031,0.56650700,-0.55242672
0.09834548,-1.1047241,3.9352294,-0.45173671,-0.54523079
-0.57996530,-0.1213315,0.1929129,0.05335482,-0.01857695
-0.32988473,-0.3302266,0.7815377,0.60647105,-0.43203380
-0.14544827,-0.2921393,-0.1770391,1.45042502,0.17375261


In [15]:
cor(ME_matrix)

,Micro_PVM,Oligo,Astro,All_GABAergic,VLMC
Micro_PVM,1.00000000,0.04802136,0.20471799,-0.0202690,0.05933915
Oligo,0.04802136,1.00000000,-0.17420110,-0.0442027,0.02980918
Astro,0.20471799,-0.17420110,1.00000000,-0.2244666,0.08411818
All_GABAergic,-0.02026900,-0.04420270,-0.22446663,1.0000000,-0.07814620
VLMC,0.05933915,0.02980918,0.08411818,-0.0781462,1.00000000


In [18]:
# 3) Fit gene-wise linear model and take residuals (+ add mean back)
X <- model.matrix(~ Micro_PVM + Oligo + Astro + All_GABAergic + VLMC, data=data.frame(ME_matrix))

fit <- lmFit(expr[,-1], X)

fitted_vals <- fit$coefficients %*% t(X)
resid_mat <- expr[,-1] - fitted_vals
expr_adj <- resid_mat # + rowMeans(expr[,-1])  # genes x sample

In [19]:
expr_adj <- data.frame(Gene=expr[,1], expr_adj)

In [20]:
dim(expr_adj)

[1] 55692   599

In [21]:
expr_adj[1:5,1:5]

,Gene,GTEX.111FC.0011.R3b.SM.GJ3PN,GTEX.117XS.0011.R3a.SM.GIN8W,GTEX.1192X.0011.R3b.SM.GIN8Y,GTEX.11DXW.0011.R3b.SM.DNZZE
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,5S_rRNA,-0.111206718,-0.086582536,0.8940152409,-0.091630419
2,5_8S_rRNA,-0.003735723,-0.002638559,-0.0004913627,-0.001632395
3,7SK,-0.025807840,-0.025131221,-0.0043044378,-0.020434042
4,A1BG,0.681855400,-0.422052587,0.7049401318,1.662233730
5,A1BG-AS1,2.339025536,-1.980647166,1.3690365977,-0.139497940


In [ ]:
fwrite(expr_adj, file=paste0("data/", data_source, "_", set_source, "_", paste(sets, collapse="_"), "_residuals_subtracted.csv"))